# alignment

> Alignment service for temporal coordination via Silero VAD plugin

In [ ]:
#| default_exp services.alignment

In [ ]:
#| export
from typing import List, Dict, Any, Optional
import asyncio

from cjm_plugin_system.core.manager import PluginManager

from cjm_transcript_vad_align.models import VADChunk

## AlignmentService

This service wraps the Silero VAD plugin to provide voice activity detection data. It processes audio files to extract time ranges where speech is detected, which are then used to align text segments with their corresponding audio positions.

In [ ]:
#| export
class AlignmentService:
    """Service for temporal alignment via Silero VAD plugin."""
    
    def __init__(
        self,
        plugin_manager: PluginManager,  # Plugin manager for accessing VAD plugin
        plugin_name: str = "cjm-media-plugin-silero-vad"  # Name of the VAD plugin
    ):
        """Initialize the alignment service."""
        self._manager = plugin_manager
        self._plugin_name = plugin_name
    
    def is_available(self) -> bool:  # True if plugin is loaded and ready
        """Check if the VAD plugin is available."""
        return self._manager.get_plugin(self._plugin_name) is not None
    
    def ensure_loaded(
        self,
        config: Optional[Dict[str, Any]] = None  # Optional plugin configuration
    ) -> bool:  # True if successfully loaded
        """Ensure the VAD plugin is loaded."""
        if self.is_available():
            return True
        
        # Try to find and load the plugin
        meta = self._manager.get_discovered_meta(self._plugin_name)
        if meta:
            return self._manager.load_plugin(meta, config or {"threshold": 0.5})
        return False
    
    async def analyze_audio_async(
        self,
        media_path: str  # Path to audio/video file
    ) -> tuple[List[VADChunk], float]:  # (VAD chunks, total duration)
        """Analyze audio file and return VAD chunks."""
        if not self.is_available():
            raise RuntimeError(f"Plugin {self._plugin_name} not loaded")
        
        # Execute plugin
        result = await self._manager.execute_plugin_async(
            self._plugin_name,
            media_path=media_path
        )
        
        # Extract ranges and metadata
        ranges = result.get('ranges', [])
        metadata = result.get('metadata', {})
        duration = metadata.get('duration', 0.0)
        
        # Convert to VADChunk objects
        chunks = []
        for idx, r in enumerate(ranges):
            chunk = VADChunk(
                index=idx,
                start_time=r.get('start', r.get('start_time', 0.0)),
                end_time=r.get('end', r.get('end_time', 0.0))
            )
            chunks.append(chunk)
        
        return chunks, duration
    
    def analyze_audio(
        self,
        media_path: str  # Path to audio/video file
    ) -> tuple[List[VADChunk], float]:  # (VAD chunks, total duration)
        """Analyze audio file synchronously."""
        return asyncio.get_event_loop().run_until_complete(
            self.analyze_audio_async(media_path)
        )

## Alignment Helpers

In [ ]:
#| export
def check_alignment_ready(
    segment_count: int,  # Number of text segments
    chunk_count: int,  # Number of VAD chunks
) -> bool:  # True if counts match and alignment can proceed
    """Check if segment and VAD chunk counts match for 1:1 alignment."""
    return segment_count > 0 and chunk_count > 0 and segment_count == chunk_count

## Tests

The following cells demonstrate the alignment service and helper functions.

In [ ]:
# Test check_alignment_ready
print(f"3 segments vs 3 chunks: {check_alignment_ready(3, 3)}")  # True
print(f"3 segments vs 5 chunks: {check_alignment_ready(3, 5)}")  # False
print(f"0 segments vs 3 chunks: {check_alignment_ready(0, 3)}")  # False
print(f"3 segments vs 0 chunks: {check_alignment_ready(3, 0)}")  # False

3 segments vs 3 chunks: True
3 segments vs 5 chunks: False
0 segments vs 3 chunks: False
3 segments vs 0 chunks: False


### AlignmentService with Silero VAD Plugin

These tests require the Silero VAD plugin to be installed and discoverable.

In [ ]:
#| eval: false
# Test AlignmentService with Silero VAD plugin
from pathlib import Path
from cjm_plugin_system.core.manager import PluginManager

# Calculate project root from notebook location (nbs/services/ -> project root)
project_root = Path.cwd().parent.parent
manifests_dir = project_root / ".cjm" / "manifests"

# Create plugin manager with explicit search path
manager = PluginManager(search_paths=[manifests_dir])
manager.discover_manifests()

print(f"Discovered {len(manager.discovered)} plugins from {manifests_dir}")

# Check if VAD plugin is available
vad_meta = manager.get_discovered_meta("cjm-media-plugin-silero-vad")
if vad_meta:
    print(f"Found plugin: {vad_meta.name} v{vad_meta.version}")
else:
    print("Silero VAD plugin not found - install via plugins.yaml")

[PluginManager] Discovered manifest: cjm-media-plugin-silero-vad from /mnt/SN850X_8TB_EXT4/Projects/GitHub/cj-mills/cjm-transcript-vad-align/.cjm/manifests/cjm-media-plugin-silero-vad.json


Discovered 1 plugins from /mnt/SN850X_8TB_EXT4/Projects/GitHub/cj-mills/cjm-transcript-vad-align/.cjm/manifests
Found plugin: cjm-media-plugin-silero-vad v0.0.2


In [ ]:
#| eval: false
# Initialize and test AlignmentService
if vad_meta:
    # Load the plugin
    manager.load_plugin(vad_meta, {"threshold": 0.5})
    
    align_service = AlignmentService(manager)
    print(f"Plugin available: {align_service.is_available()}")
    
    # Test audio analysis (use await directly - Jupyter supports top-level await)
    audio_file = "/mnt/SN850X_8TB_EXT4/Projects/GitHub/cj-mills/cjm-transcription-plugin-voxtral-hf/test_files/02 - 1. Laying Plans.mp3"
    
    chunks, duration = await align_service.analyze_audio_async(audio_file)
    
    print(f"\nAudio duration: {duration:.2f}s")
    print(f"Detected {len(chunks)} VAD chunks")
    print("\nFirst 5 chunks:")
    for chunk in chunks[:5]:
        print(f"  [{chunk.index}] {chunk.start_time:.2f}s - {chunk.end_time:.2f}s (duration: {chunk.duration:.2f}s)")

[PluginManager] Launching worker for cjm-media-plugin-silero-vad...


[cjm-media-plugin-silero-vad] Starting worker on port 52517...
[cjm-media-plugin-silero-vad] Logs: /home/innom-dt/.cjm/logs/cjm-media-plugin-silero-vad.log


[PluginManager] HTTP Request: GET http://127.0.0.1:52517/health "HTTP/1.1 200 OK"
[PluginManager] HTTP Request: POST http://127.0.0.1:52517/initialize "HTTP/1.1 200 OK"
[PluginManager] Loaded plugin: cjm-media-plugin-silero-vad
[PluginManager] HTTP Request: POST http://127.0.0.1:52517/execute "HTTP/1.1 200 OK"


[cjm-media-plugin-silero-vad] Worker ready.
Plugin available: True

Audio duration: 256.39s
Detected 89 VAD chunks

First 5 chunks:
  [0] 1.10s - 2.30s (duration: 1.20s)
  [1] 4.80s - 5.90s (duration: 1.10s)
  [2] 6.60s - 9.80s (duration: 3.20s)
  [3] 10.70s - 12.40s (duration: 1.70s)
  [4] 12.60s - 15.40s (duration: 2.80s)


In [ ]:
#| eval: false
# Cleanup
if vad_meta:
    manager.unload_all()
    print("Plugins unloaded")

[PluginManager] HTTP Request: POST http://127.0.0.1:52517/cleanup "HTTP/1.1 200 OK"
[PluginManager] Unloaded plugin: cjm-media-plugin-silero-vad


Plugins unloaded


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()